# 注意力机制（上）BiLSTM + 加性注意力

回顾 chap6 下：把情感词放在长序列开头，单向 LSTM 只有 ~50% 准确率，双向 LSTM 完美解决。但双向 LSTM 把全序列压成一个固定大小的向量——信息量大时容易丢细节。

**注意力机制**让模型在做最终决策时"按需"从所有时间步上抽取相关信息：

1. 用一个**打分函数**给每个时间步一个分数 $s_t$
2. softmax 归一化得到注意力权重 $\alpha_t$
3. 加权平均所有时间步的隐状态作为输出

本节实现**加性 (additive) 注意力**：$s_t = v^\top \tanh(W h_t)$。

## 1. 数据：情感分类（信号随机位置）

情感词放在序列**随机位置**——模型必须自己"找出"信号在哪里。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

VOCAB, PAD = 25, 0
POS = [1, 2, 3]; NEG = [4, 5, 6]; FILL = list(range(7, VOCAB))

def make_sample(g, L_range=(35, 46), n_sentiment=2):
    L = torch.randint(L_range[0], L_range[1], (1,), generator=g).item()
    label = torch.randint(0, 2, (1,), generator=g).item()
    tokens = [FILL[torch.randint(0, len(FILL), (1,), generator=g).item()] for _ in range(L)]
    sent_pool = POS if label == 1 else NEG
    positions = torch.randperm(L, generator=g)[:n_sentiment].tolist()
    for p in positions:
        tokens[p] = sent_pool[torch.randint(0, 3, (1,), generator=g).item()]
    return torch.tensor(tokens, dtype=torch.long), label

class SentimentDS(Dataset):
    def __init__(self, n, seed):
        g = torch.Generator().manual_seed(seed)
        self.data = [make_sample(g) for _ in range(n)]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    return pad_sequence(seqs, batch_first=True, padding_value=PAD), lengths, torch.tensor(labels)

train_loader = DataLoader(SentimentDS(2000, seed=0), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader   = DataLoader(SentimentDS(500,  seed=1), batch_size=64, collate_fn=collate)

## 2. 加性注意力

$$s_t = v^\top \tanh(W h_t), \quad \alpha_t = \frac{\exp(s_t)}{\sum_j \exp(s_j)}, \quad c = \sum_t \alpha_t h_t$$

对 padding 位置要 mask 掉（把分数设为 $-\infty$ 后 softmax → 权重 0）。

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size, attn_size=64):
        super().__init__()
        self.W = nn.Linear(hidden_size, attn_size, bias=False)
        self.v = nn.Linear(attn_size, 1, bias=False)

    def forward(self, H, mask):                # H: [B, L, hidden], mask: [B, L] (True=valid)
        scores = self.v(torch.tanh(self.W(H))).squeeze(-1)    # [B, L]
        scores = scores.masked_fill(~mask, -1e9)
        attn = F.softmax(scores, dim=-1)                      # [B, L]
        context = torch.bmm(attn.unsqueeze(1), H).squeeze(1)  # [B, hidden]
        return context, attn

# 快速验证
B, L, h = 2, 5, 8
H = torch.randn(B, L, h)
mask = torch.tensor([[True]*5, [True, True, True, False, False]])
att = AdditiveAttention(h, 16)
ctx, weights = att(H, mask)
print('context:', tuple(ctx.shape), '  weights:', tuple(weights.shape))
print('weights row 0 sum:', weights[0].sum().item())
print('weights row 1 at pad positions (3, 4):', weights[1, 3:].tolist())

## 3. BiLSTM + 注意力分类器

In [ ]:
class BiLSTMAttn(nn.Module):
    def __init__(self, vocab=VOCAB, embed=32, hidden=64, attn=32, n_class=2, use_attention=True):
        super().__init__()
        self.use_attention = use_attention
        self.emb = nn.Embedding(vocab, embed, padding_idx=PAD)
        self.lstm = nn.LSTM(embed, hidden, batch_first=True, bidirectional=True)
        self.attn = AdditiveAttention(hidden * 2, attn)
        self.fc = nn.Linear(hidden * 2, n_class)

    def forward(self, padded, lengths, return_attn=False):
        e = self.emb(padded)
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h, _) = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)

        if self.use_attention:
            mask = torch.arange(out.size(1))[None, :] < lengths[:, None]
            context, attn = self.attn(out, mask)
            logits = self.fc(context)
            return (logits, attn) if return_attn else logits
        else:
            feat = torch.cat([h[0], h[1]], dim=1)
            return self.fc(feat)

## 4. 训练对比：BiLSTM only vs BiLSTM + Attention

In [ ]:
def train(use_attention, epochs=5, lr=3e-3):
    torch.manual_seed(0)
    model = BiLSTMAttn(use_attention=use_attention)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    accs = []
    for _ in range(epochs):
        model.train()
        for x, l, y in train_loader:
            opt.zero_grad()
            loss_fn(model(x, l), y).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, l, y in dev_loader:
                correct += (model(x, l).argmax(1) == y).sum().item(); total += y.size(0)
        accs.append(correct / total)
    return accs, model

no_attn,   model_no   = train(use_attention=False)
with_attn, model_with = train(use_attention=True)
for ep, (n, w) in enumerate(zip(no_attn, with_attn), 1):
    print(f'epoch {ep}: bi-LSTM only={n:.4f}   bi-LSTM + attn={w:.4f}')

In [ ]:
plt.plot(no_attn,   '-o', label='BiLSTM only')
plt.plot(with_attn, '-s', label='BiLSTM + additive attn')
plt.xlabel('epoch'); plt.ylabel('dev accuracy'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. 可视化注意力权重

看一个例子：模型在做分类时把注意力集中在哪些位置？理想情况是聚焦在情感词上。

In [ ]:
x_batch, len_batch, y_batch = next(iter(dev_loader))
# 找一条 POS / NEG 信号清晰的样本
model_with.eval()
x_one, len_one = x_batch[:1], len_batch[:1]
with torch.no_grad():
    logits, attn = model_with(x_one, len_one, return_attn=True)

tokens = x_one[0, :len_one[0]].tolist()
weights = attn[0, :len_one[0]].tolist()

def token_kind(t):
    if t in POS: return 'POS'
    if t in NEG: return 'NEG'
    return '...'

print(f'true={y_batch[0].item()}   pred={logits.argmax(1).item()}')
print()
print(f"{'idx':>3}  {'tok':>3}  {'kind':>4}  attn_weight")
for i, (t, w) in enumerate(zip(tokens, weights)):
    bar = '#' * int(w * 60)
    print(f'{i:>3}  {t:>3}  {token_kind(t):>4}  {w:.4f}  {bar}')

注意力权重应当在 POS / NEG 的位置显著高于普通 filler——模型学会了"挑"出关键 token。

## 小结

- **加性注意力**：$s_t = v^\top \tanh(W h_t)$，可学习参数 $W$、$v$。
- **mask** padding 位置：把得分设为 $-\infty$ 让 softmax 自然给出 0 权重——不要先 softmax 再乘 mask（破坏归一化）。
- **`torch.bmm(attn.unsqueeze(1), H)`** 一步算出加权平均：`[B, 1, L] × [B, L, hidden]` → `[B, 1, hidden]`。
- 注意力让模型**显式**学习"哪里重要"，可解释性好——可以画 heatmap 看模型在看什么。
- chap8 下：自注意力机制 + Transformer Encoder。